## metadata process notebook

In [1]:
import plotly.graph_objects as go
import polars as pl
import matplotlib.pyplot as plt
import plotly.colors as pc
import seaborn as sns
import kaleido

In [2]:
rice_meta_data = pl.read_csv(
    "../Data/01_Metadata/HS_rice_meta-data.csv", 
    separator = ","
).with_columns(
    pl.lit("FASTQ data pairs").alias("data type")
)


display(rice_meta_data.head())

All-pair,Project-pair,Organism,Sub-species,Cultivar,Genotype,Simple stress condition,SRP accession,GEO Accession,Stress,Control,Library_Layout,Stress temperature (day/night) (℃),Control temperature (day/night)(℃),Time,Heat recovery,Treatment condition,Tissue,Period,Instrument,LibrarySelection,DOI,GSM_Pair_name,note,Technical Note,data type
i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
1,1,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741460""","""SRR22741464""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
2,2,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741461""","""SRR22741465""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
3,3,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741462""","""SRR22741466""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
4,4,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741463""","""SRR22741467""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
5,5,"""O.sativa""","""ssp. japonica""","""Cypress""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741468""","""SRR22741472""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""Cypress(HNT-sensitive), high-n…",null,"""FASTQ data pairs"""


In [3]:
def create_sankey_from_polars(df, flow_columns, title="Sankey Diagram", column_color_map=None, node_align="justify", width=1500, height=1000):
    """
    Function to create Sankey plot from Polars DataFrame (improved label version)
    
    Parameters:
    df: Polars DataFrame
    flow_columns: list - list of columns to represent the flow (from left to right)
    title: str - title of the graph
    column_color_map: dict - column name as key, color as value (e.g., {'col1': 'blue', 'col2': 'red'})
    node_align: str - alignment of nodes within each column. One of 'left', 'right', 'center', 'justify'. Default is 'justify'.
    """
    
    # data preparation
    all_nodes = []
    source_list = []
    target_list = []
    value_list = []
    node_to_column = {}
    
    # create flow for each step
    for i in range(len(flow_columns) - 1):
        source_col = flow_columns[i]
        target_col = flow_columns[i + 1]
        
        # group by and aggregate
        flow_data = (df
                    .group_by([source_col, target_col])
                    .agg(pl.len().alias("count"))
                    .filter(pl.col("count") > 0)  # exclude zero values
                    )
        
        # add step number to source and target to make them unique
        for row in flow_data.iter_rows(named=True):
            source_label = f"{row[source_col]}_S{i}"
            target_label = f"{row[target_col]}_S{i+1}"
            
            if source_label not in all_nodes:
                all_nodes.append(source_label)
                node_to_column[source_label] = source_col
            if target_label not in all_nodes:
                all_nodes.append(target_label)
                node_to_column[target_label] = target_col
                
            source_list.append(all_nodes.index(source_label))
            target_list.append(all_nodes.index(target_label))
            value_list.append(row["count"])
    
    # remove _S + number from label
    def clean_label(label):
        # use regular expression to remove _S + number from label
        import re
        return re.sub(r'_S\d+$', '', label)
    
    # color setting
    node_colors = []
    default_color = "rgba(201, 203, 207, 0.8)"  # default color

    if column_color_map is None:
        column_color_map = {}

    for node in all_nodes:
        column_name = node_to_column.get(node)
        color = column_color_map.get(column_name, default_color)
        node_colors.append(color)
    
    # create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=[clean_label(node) for node in all_nodes],  # use clean label
            color=node_colors,
            align=node_align
        ),
        link=dict(
            source=source_list,
            target=target_list,
            value=value_list,
            color="rgba(128, 128, 128, 0.4)"  # link color
        )
    )])
    
    fig.update_layout(
        title_text=title,
        font_size=13,
        width=width,
        height=height
        
    )
    
    return fig

In [4]:
config = {
  'toImageButtonOptions': {
    'format': 'png', # png, svg, jpeg, webpから選択
    'filename': 'sankey_fig1',
    'width': 2400,
    'height': 800,
    'scale': 3 # 解像度を3倍にする (この値を大きくすると高解像度になります)
  }
}

my_colors = {
    "data type": "cornflowerblue",
    "GEO Accession": "darkblue",
    "Sub-species": "seagreen",
    "Cultivar": "darkolivegreen",
    "Tissue": "darkorange",
    "Genotype": "darkmagenta"
}

sankey_fig1 = create_sankey_from_polars(
    rice_meta_data,
    ["data type", "GEO Accession", "Sub-species", "Cultivar", "Tissue", "Genotype"],
    "heat stress-related bulk RNA-seq metadata in rice (sample level)",
    column_color_map=my_colors,
    node_align="right",
    width=2400,
    height=800
)
sankey_fig1.show(config=config)

In [5]:
display(rice_meta_data.head())

All-pair,Project-pair,Organism,Sub-species,Cultivar,Genotype,Simple stress condition,SRP accession,GEO Accession,Stress,Control,Library_Layout,Stress temperature (day/night) (℃),Control temperature (day/night)(℃),Time,Heat recovery,Treatment condition,Tissue,Period,Instrument,LibrarySelection,DOI,GSM_Pair_name,note,Technical Note,data type
i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
1,1,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741460""","""SRR22741464""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
2,2,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741461""","""SRR22741465""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
3,3,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741462""","""SRR22741466""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
4,4,"""O.sativa""","""ssp. japonica""","""Lagrue""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741463""","""SRR22741467""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""LaGrue(HNT-tolerant), high-nig…",null,"""FASTQ data pairs"""
5,5,"""O.sativa""","""ssp. japonica""","""Cypress""","""WT""","""HNT_30C_28C_10h""","""SRP413097""","""GSE220996""","""SRR22741468""","""SRR22741472""","""PAIRED""","""30/28""","""30/22.2""","""10h""",null,"""green house""","""seed, endosperm, R6 caryopsis""","""R2 stage (booting) - R6 stage …","""Illumina HiSeq 2000""","""cDNA""","""10.1038/s41598-023-31399-w""",null,"""Cypress(HNT-sensitive), high-n…",null,"""FASTQ data pairs"""


In [6]:
config2 = {
  'toImageButtonOptions': {
    'format': 'png', # png, svg, jpeg, webpから選択
    'filename': 'sankey_fig2',
    'width': 2400,
    'height': 800,
    'scale': 3 # 解像度を3倍にする (この値を大きくすると高解像度になります)
  }
}

my_colors = {
    "data type": "cornflowerblue",
    "GEO Accession": "darkblue",
    "Sub-species": "seagreen",
    "Simple stress condition": "darkred",
    "Treatment condition": "slategray"
}

sankey_fig1 = create_sankey_from_polars(
    rice_meta_data,
    ["data type", "GEO Accession", "Sub-species", "Simple stress condition", "Treatment condition"],
    "heat stress-related bulk RNA-seq metadata in rice (sample level)",
    column_color_map=my_colors,
    node_align="right",
    width=2400,
    height=800
)
sankey_fig1.show(config=config2)

In [7]:
print(rice_meta_data.select(pl.col("SRP accession").n_unique()))
print(rice_meta_data.select(pl.col("Cultivar").n_unique()))
print(rice_meta_data.select(pl.col("Treatment condition").n_unique()))
print(rice_meta_data.select(pl.col("Tissue").n_unique()))

shape: (1, 1)
┌───────────────┐
│ SRP accession │
│ ---           │
│ u32           │
╞═══════════════╡
│ 13            │
└───────────────┘
shape: (1, 1)
┌──────────┐
│ Cultivar │
│ ---      │
│ u32      │
╞══════════╡
│ 18       │
└──────────┘
shape: (1, 1)
┌─────────────────────┐
│ Treatment condition │
│ ---                 │
│ u32                 │
╞═════════════════════╡
│ 9                   │
└─────────────────────┘
shape: (1, 1)
┌────────┐
│ Tissue │
│ ---    │
│ u32    │
╞════════╡
│ 15     │
└────────┘


In [8]:
rice_meta_data_extract_group = rice_meta_data.group_by(
    ['Sub-species']
).agg(
    pl.col("SRP accession").n_unique().alias("SRP_accession_count"),
    pl.col("Tissue").n_unique().alias("Tissue_count")
)

display(rice_meta_data_extract_group)

Sub-species,SRP_accession_count,Tissue_count
str,u32,u32
"""no description""",1,1
"""ssp. japonica""",10,13
"""ssp. indica""",5,7


&nbsp;

&nbsp;

&nbsp;

## Separate dataframe based on read type

In [9]:
# rice_meta_data_se = rice_meta_data.filter(pl.col('Library_Layout') == 'SINGLE')
# rice_meta_data_se.write_csv('../Data/01_Meta-data/HS_rice_meta-data_SE.csv', separator=',')

# rice_meta_data_pe = rice_meta_data.filter(pl.col('Library_Layout') == 'PAIRED')
# rice_meta_data_pe.write_csv('../Data/01_Meta-data/HS_rice_meta-data_PE.csv', separator=',')

# display(rice_meta_data_se.head(), rice_meta_data_pe.head())